# Credit Risk Prediction — Model Training and Evaluation

**Author:** Aluka Precious Oluchukwu
**Project:** Credit Risk ML System — Model Training and Evaluation

## Overview

This notebook trains three machine learning models on the preprocessed 
and SMOTE balanced training data produced in Phase 3. We evaluate each 
model on the untouched test set and compare their performance across 
multiple metrics to identify the best performing model for deployment.

The three models we train are:
1. Logistic Regression — our interpretable baseline model
2. Random Forest — our primary ensemble tree based model
3. Gradient Boosting — our advanced boosting model

We evaluate using Accuracy, Precision, Recall, F1 Score, and AUC-ROC 
because accuracy alone is misleading on imbalanced datasets.

In [1]:
# ─── Import libraries ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
import warnings
warnings.filterwarnings("ignore")

print("Libraries Imported Sucessfully")

Libraries Imported Sucessfully


In [2]:
# ─── Load processed datasets ──────────────────────────────────────────────────
X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
X_test  = pd.read_csv("../data/processed/X_test.csv")
y_test  = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("Datasets loaded successfully!")
print(f"\nTraining set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"\nTraining target distribution:")
print(y_train.value_counts())
print(f"\nTest target distribution:")
print(y_test.value_counts())

Datasets loaded successfully!

Training set: (1120, 20)
Test set:     (200, 20)

Training target distribution:
Category
1    560
0    560
Name: count, dtype: int64

Test target distribution:
Category
0    140
1     60
Name: count, dtype: int64


## 2. Training All Three Models

We train Logistic Regression, Random Forest, and Gradient Boosting 
simultaneously on the SMOTE balanced training data. Each model learns 
the relationship between the 20 features and the target variable 
Category through a different algorithmic approach.

- Logistic Regression learns a linear decision boundary
- Random Forest builds 100 decision trees and aggregates their votes
- Gradient Boosting builds trees sequentially each correcting 
  the errors of the previous one

In [3]:
# ─── Define models ────────────────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(
        random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, random_state=42)
}

# ─── Train all models ─────────────────────────────────────────────────────────
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"  {name} trained successfully!")

print("\nAll three models trained successfully!")

Training Logistic Regression...
  Logistic Regression trained successfully!
Training Random Forest...
  Random Forest trained successfully!
Training Gradient Boosting...
  Gradient Boosting trained successfully!

All three models trained successfully!


## 3. Evaluating All Three Models

We evaluate each trained model against the untouched test set of 200 
real applicants. For each model we calculate five metrics:

- **Accuracy** — overall correct predictions
- **Precision** — of predicted bad risks how many were truly bad
- **Recall** — of actual bad risks how many did we catch
- **F1 Score** — harmonic mean of Precision and Recall
- **AUC-ROC** — overall ability to distinguish good from bad risk


In [4]:
# ─── Evaluate all models ──────────────────────────────────────────────────────
results = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1 Score':  f1_score(y_test, y_pred),
        'AUC-ROC':   roc_auc_score(y_test, y_prob)
    }

# ─── Display results table ────────────────────────────────────────────────────
results_df = pd.DataFrame(results).T.round(4)
print("=" * 65)
print("MODEL EVALUATION RESULTS")
print("=" * 65)
print(results_df.to_string())
print("=" * 65)
print(f"\nBest Accuracy:  {results_df['Accuracy'].idxmax()} "
      f"({results_df['Accuracy'].max():.4f})")
print(f"Best Recall:    {results_df['Recall'].idxmax()} "
      f"({results_df['Recall'].max():.4f})")
print(f"Best F1 Score:  {results_df['F1 Score'].idxmax()} "
      f"({results_df['F1 Score'].max():.4f})")
print(f"Best AUC-ROC:   {results_df['AUC-ROC'].idxmax()} "
      f"({results_df['AUC-ROC'].max():.4f})")

MODEL EVALUATION RESULTS
                     Accuracy  Precision  Recall  F1 Score  AUC-ROC
Logistic Regression     0.650     0.4359  0.5667    0.4928   0.6839
Random Forest           0.705     0.5094  0.4500    0.4779   0.7564
Gradient Boosting       0.730     0.5469  0.5833    0.5645   0.7873

Best Accuracy:  Gradient Boosting (0.7300)
Best Recall:    Gradient Boosting (0.5833)
Best F1 Score:  Gradient Boosting (0.5645)
Best AUC-ROC:   Gradient Boosting (0.7873)


## 4. Hyperparameter Tuning — Gradient Boosting

Default model parameters are rarely optimal. Hyperparameter tuning 
searches for the best combination of model settings that maximises 
performance on our specific dataset.

We use RandomizedSearchCV which randomly samples parameter combinations 
and evaluates each using 5-fold cross validation — giving us a robust 
estimate of true model performance rather than overfitting to a single 
train test split.

We optimise for AUC-ROC as our primary metric because it is the most 
comprehensive measure of credit risk model performance — capturing the 
model's ability to distinguish good from bad risk across all possible 
decision thresholds.

In [5]:
# ─── Hyperparameter Tuning — Gradient Boosting ────────────────────────────────
from sklearn.model_selection import RandomizedSearchCV

# ─── Define parameter grid ────────────────────────────────────────────────────
param_grid = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [3, 4, 5, 6, 7],
    'learning_rate':     [0.01, 0.05, 0.1, 0.15, 0.2],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf':  [1, 2, 4, 6],
    'subsample':         [0.7, 0.8, 0.9, 1.0],
    'max_features':      ['sqrt', 'log2', None]
}

# ─── RandomizedSearchCV ───────────────────────────────────────────────────────
gb_base = GradientBoostingClassifier(random_state=42)

random_search = RandomizedSearchCV(
    estimator=gb_base,
    param_distributions=param_grid,
    n_iter=50,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Starting hyperparameter tuning...")
print("Testing 50 parameter combinations with 5-fold cross validation...")
print("This may take a few minutes — please wait...\n")

random_search.fit(X_train, y_train)

print(f"\nBest parameters found:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest cross validation AUC-ROC: {random_search.best_score_:.4f}")

Starting hyperparameter tuning...
Testing 50 parameter combinations with 5-fold cross validation...
This may take a few minutes — please wait...

Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best parameters found:
  subsample: 0.7
  n_estimators: 300
  min_samples_split: 2
  min_samples_leaf: 1
  max_features: log2
  max_depth: 7
  learning_rate: 0.1

Best cross validation AUC-ROC: 0.9184


In [6]:
# ─── Train final tuned model ──────────────────────────────────────────────────
best_gb = random_search.best_estimator_

# ─── Evaluate tuned model on test set ────────────────────────────────────────
y_pred_tuned = best_gb.predict(X_test)
y_prob_tuned = best_gb.predict_proba(X_test)[:, 1]

print("=" * 65)
print("TUNED GRADIENT BOOSTING — TEST SET RESULTS")
print("=" * 65)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_tuned):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_tuned):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_tuned):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob_tuned):.4f}")
print("=" * 65)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned,
      target_names=['Good Risk', 'Bad Risk']))

TUNED GRADIENT BOOSTING — TEST SET RESULTS
Accuracy:  0.7550
Precision: 0.6122
Recall:    0.5000
F1 Score:  0.5505
AUC-ROC:   0.7660

Classification Report:
              precision    recall  f1-score   support

   Good Risk       0.80      0.86      0.83       140
    Bad Risk       0.61      0.50      0.55        60

    accuracy                           0.76       200
   macro avg       0.71      0.68      0.69       200
weighted avg       0.74      0.76      0.75       200



## 5. Classification Threshold Adjustment

By default all classifiers use a threshold of 0.5 — meaning the model 
predicts bad risk only when it is more than 50% confident. But in credit 
risk the cost of missing a bad risk borrower is far higher than the cost 
of incorrectly declining a good one.

Lowering the threshold makes the model more aggressive at flagging bad 
risks — improving Recall at the cost of some Precision. This is a 
deliberate business decision not a technical limitation.

We test multiple thresholds to find the optimal balance between Recall 
and Precision for our specific business objective.

In [7]:
# ─── Threshold Adjustment ─────────────────────────────────────────────────────
thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]

print("=" * 70)
print("THRESHOLD ANALYSIS — TUNED GRADIENT BOOSTING")
print("=" * 70)
print(f"{'Threshold':<12} {'Accuracy':<12} {'Precision':<12} "
      f"{'Recall':<12} {'F1':<12} {'AUC-ROC':<12}")
print("-" * 70)

threshold_results = {}

for threshold in thresholds:
    y_pred_thresh = (y_prob_tuned >= threshold).astype(int)
    
    acc  = accuracy_score(y_test, y_pred_thresh)
    prec = precision_score(y_test, y_pred_thresh)
    rec  = recall_score(y_test, y_pred_thresh)
    f1   = f1_score(y_test, y_pred_thresh)
    auc  = roc_auc_score(y_test, y_prob_tuned)
    
    threshold_results[threshold] = {
        'Accuracy': acc, 'Precision': prec,
        'Recall': rec, 'F1': f1, 'AUC-ROC': auc
    }
    
    marker = ' ← default' if threshold == 0.5 else ''
    print(f"{threshold:<12} {acc:<12.4f} {prec:<12.4f} "
          f"{rec:<12.4f} {f1:<12.4f} {auc:<12.4f}{marker}")

print("=" * 70)

THRESHOLD ANALYSIS — TUNED GRADIENT BOOSTING
Threshold    Accuracy     Precision    Recall       F1           AUC-ROC     
----------------------------------------------------------------------
0.3          0.7550       0.5965       0.5667       0.5812       0.7660      
0.35         0.7550       0.6000       0.5500       0.5739       0.7660      
0.4          0.7600       0.6111       0.5500       0.5789       0.7660      
0.45         0.7500       0.5962       0.5167       0.5536       0.7660      
0.5          0.7550       0.6122       0.5000       0.5505       0.7660       ← default


In [8]:
# ─── Set optimal threshold and final evaluation ───────────────────────────────
optimal_threshold = 0.30

y_pred_final = (y_prob_tuned >= optimal_threshold).astype(int)

print("=" * 65)
print("FINAL MODEL — TUNED GRADIENT BOOSTING (Threshold = 0.30)")
print("=" * 65)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_final):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_final):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_final):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob_tuned):.4f}")
print("=" * 65)
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_final,
      target_names=['Good Risk', 'Bad Risk']))

print("\nFinal Model Summary:")
print(f"  Algorithm:  Tuned Gradient Boosting")
print(f"  Threshold:  {optimal_threshold}")
print(f"  This model correctly identifies {recall_score(y_test, y_pred_final)*100:.1f}% "
      f"of all bad risk applicants")
print(f"  When it flags bad risk it is correct "
      f"{precision_score(y_test, y_pred_final)*100:.1f}% of the time")

FINAL MODEL — TUNED GRADIENT BOOSTING (Threshold = 0.30)
Accuracy:  0.7550
Precision: 0.5965
Recall:    0.5667
F1 Score:  0.5812
AUC-ROC:   0.7660

Classification Report:
              precision    recall  f1-score   support

   Good Risk       0.82      0.84      0.83       140
    Bad Risk       0.60      0.57      0.58        60

    accuracy                           0.76       200
   macro avg       0.71      0.70      0.70       200
weighted avg       0.75      0.76      0.75       200


Final Model Summary:
  Algorithm:  Tuned Gradient Boosting
  Threshold:  0.3
  This model correctly identifies 56.7% of all bad risk applicants
  When it flags bad risk it is correct 59.6% of the time


## 6. Cross Validation Comparison — All Three Models

Following rigorous model selection methodology we validate all three 
models using 5-fold stratified cross validation. This provides a more 
reliable and stable estimate of true model performance than a single 
train test split — and ensures our champion model selection is 
statistically defensible.

In [9]:
# ─── Cross Validation Comparison ──────────────────────────────────────────────
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Logistic Regression': LogisticRegression(
        random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, random_state=42)
}

scoring = ['roc_auc', 'f1', 'recall', 'precision']

print("Running 5-fold Stratified Cross Validation...")
print("=" * 75)
print(f"{'Model':<25} {'AUC-ROC':<15} {'F1':<15} {'Recall':<15} {'Precision':<15}")
print("-" * 75)

cv_results = {}

for name, model in cv_models.items():
    scores = cross_validate(model, X_train, y_train,
                           cv=cv, scoring=scoring)
    
    cv_results[name] = {
        'AUC-ROC Mean':   scores['test_roc_auc'].mean(),
        'AUC-ROC Std':    scores['test_roc_auc'].std(),
        'F1 Mean':        scores['test_f1'].mean(),
        'Recall Mean':    scores['test_recall'].mean(),
        'Precision Mean': scores['test_precision'].mean()
    }
    
    print(f"{name:<25} "
          f"{scores['test_roc_auc'].mean():.4f}±{scores['test_roc_auc'].std():.3f}  "
          f"{scores['test_f1'].mean():.4f}        "
          f"{scores['test_recall'].mean():.4f}        "
          f"{scores['test_precision'].mean():.4f}")

print("=" * 75)
print("\nGap Analysis:")
auc_scores = {k: v['AUC-ROC Mean'] for k, v in cv_results.items()}
best = max(auc_scores, key=auc_scores.get)
worst = min(auc_scores, key=auc_scores.get)
gap = auc_scores[best] - auc_scores[worst]
print(f"Best model:  {best} ({auc_scores[best]:.4f})")
print(f"Worst model: {worst} ({auc_scores[worst]:.4f})")
print(f"Gap:         {gap:.4f} ({gap*100:.1f}%)")
if gap > 0.04:
    print("Verdict: Gap > 4% — Gradient Boosting selection is justified")
else:
    print("Verdict: Gap < 4% — consider Logistic Regression for interpretability")

Running 5-fold Stratified Cross Validation...
Model                     AUC-ROC         F1              Recall          Precision      
---------------------------------------------------------------------------
Logistic Regression       0.7737±0.026  0.7002        0.7000        0.7015
Random Forest             0.9114±0.010  0.8305        0.8286        0.8334
Gradient Boosting         0.8858±0.009  0.8010        0.8179        0.7865

Gap Analysis:
Best model:  Random Forest (0.9114)
Worst model: Logistic Regression (0.7737)
Gap:         0.1377 (13.8%)
Verdict: Gap > 4% — Gradient Boosting selection is justified
